# Reproduce NatComm Resubmission Plots

This notebook reproduces the three PNG plots referenced by `NatComm_final.tex` and records the parameter choices needed to regenerate the raw data.

There are two paths:

1. **Published-data path**: recreates the figures immediately from the numeric values present in the current repository.
2. **Raw-regeneration path**: reruns the Monte Carlo simulations with explicit seeds, sample counts, locality parameters, and Hamiltonian conditioning parameter `c`. These cells are opt-in because the full mode sweep is expensive.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "qgl_matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Make the helper import work from either the repo root or NatCommResub/.
cwd = Path.cwd().resolve()
for candidate in [cwd, cwd.parent, cwd / "NatCommResub", cwd.parent / "NatCommResub"]:
    if (candidate / "qgl_reproduce.py").exists():
        sys.path.insert(0, str(candidate))
        break

import qgl_reproduce as qgl

repo_root = qgl.find_repo_root(cwd)
manuscript_dir = repo_root / "NatCommResub"
data_dir = repo_root / "reproduction_data"
plot_dir = repo_root / "reproduced_plots"
data_dir.mkdir(exist_ok=True)
plot_dir.mkdir(exist_ok=True)

# Toggle these to True to regenerate raw simulation data instead of using published cached data.
RUN_EXPENSIVE_MODE_SWEEPS = False
RUN_EXPENSIVE_L_SWEEP = False

print(f"Repository root: {repo_root}")
print(f"Data directory:   {data_dir}")
print(f"Plot directory:   {plot_dir}")

## Parameter Choices

The scripts implement the 1D Hamiltonian through the normal chain precision matrix: a tridiagonal matrix with diagonal entries `2` except the last boundary diagonal equal to `1`, and off-diagonal entries `-1`. With inverse temperature `beta=0.5`, this is equivalent to the manuscript convention `H = beta * (NormalPrecision + c I)`.

For the mode-scaling figure the manuscript caption states `N=10^4`, locality `l=3`, and five sample realizations. The old `globalvslocal.jl` currently has `num_averaging_runs=3`, so this notebook pins the caption value `repeats=5` for new raw runs.

In [ ]:
mode_configs = {
    "ill": qgl.ModeSweepConfig(
        condition="ill",
        c=0.0,
        samples=10_000,
        locality=3,
        repeats=5,
        beta=0.5,
        seed=2024052701,
        m_values=tuple(range(100, 1151, 50)),
    ),
    "well": qgl.ModeSweepConfig(
        condition="well",
        c=0.1,
        samples=10_000,
        locality=3,
        repeats=5,
        beta=0.5,
        seed=2024052702,
        m_values=tuple(range(100, 1151, 50)),
    ),
}

l_config = qgl.LSweepConfig(
    c=0.1,
    m=100,
    sample_counts=(10_000, 100_000),
    l_values=(2, 4, 6, 8, 10),
    repeats=1,
    beta=0.5,
    seed=2024052703,
    reuse_samples_across_l=True,
)

pd.DataFrame([
    {"figure": "mode_sweep_ill", **mode_configs["ill"].__dict__},
    {"figure": "mode_sweep_well", **mode_configs["well"].__dict__},
    {"figure": "l_sweep", **l_config.__dict__},
])

## Numerical Sanity Checks

These checks verify the core exact formulas before any sampling is used: the measurement covariance is positive, satisfies the Gaussian uncertainty condition, and exact global inversion recovers the Hamiltonian to machine precision.

In [ ]:
rows = []
for m in [10, 100]:
    sigma = qgl.measurement_covariance(m, c=0.1, beta=0.5)
    quantum_cov = sigma - np.eye(2 * m) / 2
    omega = qgl.omega_matrix(m)
    rows.append({
        "m": m,
        "min_eig_measurement_cov": np.linalg.eigvalsh(sigma).min(),
        "min_eig_quantum_cov_plus_iOmega_over_2": np.linalg.eigvalsh(quantum_cov + 1j * omega / 2).min().real,
        "exact_global_error": qgl.global_reconstruction_error(sigma, m, c=0.1, beta=0.5),
        "classical_exact_error": qgl.classical_exact_error(m, c=0.1, beta=0.5),
    })
pd.DataFrame(rows)

## Persist Published Tables

The current repository contains the ill-conditioned mode-sweep table and the locality-sweep table as hard-coded Python data. This cell writes those values to CSV so future plotting does not depend on scraping Python source files.

In [ ]:
qgl.write_published_tables(data_dir)
published_files = sorted(data_dir.glob("*_published.csv"))
for path in published_files:
    print(path.relative_to(repo_root))

## Mode-Sweep Data

Set `RUN_EXPENSIVE_MODE_SWEEPS = True` in the first cell to regenerate both the ill- and well-conditioned CSVs. The generated files are written incrementally, so an interrupted run still leaves partial data in root-level `reproduction_data/`.

In [ ]:
ill_generated_csv = data_dir / "mode_sweep_ill_generated.csv"
well_generated_csv = data_dir / "mode_sweep_well_generated.csv"

if RUN_EXPENSIVE_MODE_SWEEPS:
    ill_df = qgl.run_mode_sweep(mode_configs["ill"], ill_generated_csv)
    well_df = qgl.run_mode_sweep(mode_configs["well"], well_generated_csv)
else:
    ill_df = qgl.published_ill_mode_df()
    well_df = pd.read_csv(well_generated_csv) if well_generated_csv.exists() else None

display(Markdown(f"Ill-conditioned rows available: **{len(ill_df)}**"))
if well_df is None:
    display(Markdown("Well-conditioned raw CSV is not present yet. The notebook will display/copy the existing manuscript PNG for that panel."))
else:
    display(Markdown(f"Well-conditioned rows available: **{len(well_df)}**"))

In [ ]:
ill_plot = plot_dir / "simulation_errors_plot_ill.png"
qgl.plot_mode_sweep(ill_df, "ill-conditioned", ill_plot)
plt.close("all")
display(Image(filename=str(ill_plot)))

well_plot = plot_dir / "simulation_errors_plot_well.png"
if well_df is None:
    qgl.copy_existing_plot(manuscript_dir, "simulation_errors_plot_well.png", plot_dir)
else:
    qgl.plot_mode_sweep(well_df, "well-conditioned", well_plot)
    plt.close("all")
display(Image(filename=str(well_plot)))

## Locality-Sweep Data

Set `RUN_EXPENSIVE_L_SWEEP = True` in the first cell to regenerate the sampled `N=10^4` and `N=10^5` curves and the exact-covariance inset. Without that flag, the notebook uses the published values from `plotterl.py` so the PNG is reproduced immediately.

The published exact inset values are kept as published. The current `exact_covariance.jl` gives a different value at `l=4`, which indicates that the script was changed after the plotted values were copied or that an intermediate parameter/stencil choice was used.

In [ ]:
l_sampled_csv = data_dir / "l_sweep_sampled_generated.csv"
l_exact_csv = data_dir / "l_sweep_exact_generated.csv"

if RUN_EXPENSIVE_L_SWEEP:
    sampled_l_df = qgl.run_l_sweep(l_config, l_sampled_csv)
    exact_l_df = qgl.exact_l_sweep(l_config)
    exact_l_df.to_csv(l_exact_csv, index=False)

    rng = np.random.default_rng(l_config.seed + 99)
    global_errors = []
    for _ in range(l_config.repeats):
        samples = qgl.sample_measurements(l_config.m, 100_000, l_config.c, rng, l_config.beta)
        cov_est = qgl.covariance_estimate(samples)
        global_errors.append(qgl.global_reconstruction_error(cov_est, l_config.m, l_config.c, l_config.beta))
    global_sampled_error = float(np.mean(global_errors))
    classical_exact_error = qgl.classical_exact_error(l_config.m, l_config.c, l_config.beta)
else:
    sampled_l_df = qgl.published_l_sweep_df()
    exact_l_df = qgl.published_exact_l_df()
    global_sampled_error = 0.09187648542918603
    classical_exact_error = 1.07

display(sampled_l_df)
display(exact_l_df)

In [ ]:
l_plot = plot_dir / "improved_plot.png"
qgl.plot_l_sweep(sampled_l_df, exact_l_df, global_sampled_error, classical_exact_error, l_plot)
plt.close("all")
display(Image(filename=str(l_plot)))

## Ill/Well First-Mode Window Heatmaps

This compact check reruns the first five modes for the ill-conditioned chain (`c = 0`) and the well-conditioned chain (`c = 0.1`) with the same seed. The side-by-side heatmaps make the condition-dependent reconstruction differences visible without storing the full mode-sweep matrix dump.


In [ ]:
import compare_condition_window_heatmaps as condition_windows

condition_window_summary = condition_windows.run_condition_window_checks(
    seed=100,
    m=100,
    samples=100_000,
    beta=0.5,
    locality=4,
    window_modes=5,
    start_mode=1,
    output_dir=repo_root / "condition_window_checks",
)

condition_metrics = pd.DataFrame([
    {
        "condition": item["condition"],
        "c": item["c"],
        "local_max_abs_error": item["local_metrics"]["max_abs_error"],
        "global_max_abs_error": item["global_metrics"]["max_abs_error"],
        "target_max_abs": item["local_metrics"]["target_max_abs"],
    }
    for item in condition_window_summary["conditions"]
])
display(condition_metrics)

display(Image(filename=condition_window_summary["error_heatmap"]))
display(Image(filename=condition_window_summary["target_heatmap"]))


## Raw Data Still Needed for Bitwise Reproduction

The notebook can regenerate clean raw data with fixed seeds. To reproduce the exact historical PNGs from raw data rather than from the published hard-coded values, the only missing artifact is the well-conditioned mode-sweep table behind `simulation_errors_plot_well.png`. If you have the old output, place it at root-level `reproduction_data/mode_sweep_well_generated.csv` with columns `SystemSize_m, Avg_Naive_Error, Avg_Local_Error, Avg_Naive_Time_sec, Avg_Local_Time_sec`.

If exact historical Monte Carlo values are required, the original random seeds are also needed; otherwise rerunning the notebook with the pinned seeds gives a clean, reproducible new data set.